# EDA данных для классификации типов комнат
Проверяем:

- структуру `train_df.csv`, `val_df.csv`, `test_df.csv`
- пропуски в колонках
- класс `18`
- дубликаты
- распределение классов

После этого можно осознанно упростить или изменить `src/preprocess_data.py`.

In [1]:
import pandas as pd
from pathlib import Path

In [2]:
ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR = ROOT_DIR / "data" / "raw"

csv_paths = {
    "train": RAW_DIR / "train_df.csv",
    "val": RAW_DIR / "val_df.csv",
    "test": RAW_DIR / "test_df.csv",
}

image_roots = {
    "train": RAW_DIR / "train_images",
    "val": RAW_DIR / "val_images",
    "test": RAW_DIR / "test_images",
}

train_df = pd.read_csv(csv_paths["train"])
val_df = pd.read_csv(csv_paths["val"])
test_df = pd.read_csv(csv_paths["test"])

dfs = {
    "train": train_df,
    "val": val_df,
    "test": test_df
}
dfs.keys()

dict_keys(['train', 'val', 'test'])

## 1. Признаки в датасетах

Выводим признаки так, чтобы было видно полный состав каждого датасета.


In [3]:
max_columns_count = max(len(df.columns) for df in dfs.values())

columns_view = pd.DataFrame({
    split: list(df.columns) + [""] * (max_columns_count - len(df.columns))
    for split, df in dfs.items()
})

columns_view


,train,val,test
0,item_id,item_id,image_id_ext
1,image,image,item_id
2,image_id_ext,image_id_ext,perform_top_microcat_name
3,result,result,perform_top_microcat_prob
4,label,label,perform_top_other_classes_value
5,ratio,ratio,perform_top_other_classes_prob
6,,,image


`train` и `val` содержат разметку, а `test` содержит дополнительные признаки, бесполезные для нашей задачи. `result` в `test` отсутствует

## 2. Что оставляем для пайплайна

Для текущего image-only пайплайна модель получает информацию из локальной картинки, а не из табличных признаков. Поэтому:

- в `train` и `val` оставляем `image_id_ext`, `image`, `result`
- в `test` оставляем `image_id_ext`, `image`, `item_id`
- `image_id_ext` нужен для поиска локального файла
- `image` оставляем как справочную ссылку на исходный URL
- `item_id` нужен в `test` как связь с исходной строкой при формировании предсказаний
- `label` убираем
- `ratio` убираем
- `perform_top_*` убираем из `test`, так как они не являются информативными для нашего кейса

Для `train` и `val` также удаляем строки, которые нельзя использовать для обучения или оценки: нет `image_id_ext`, некорректный `result`, класс `18`, отсутствует локальный файл изображения. Для `test` строки не удаляем, потому что внешняя проверка может ожидать предсказание для каждой исходной строки.


In [4]:
selected_columns = {
    "train": ["image_id_ext", "image", "result"],
    "val": ["image_id_ext", "image", "result"],
    "test": ["image_id_ext", "image", "item_id"],
}

columns_to_drop_report = []
for split, df in dfs.items():
    columns_to_drop = [column for column in df.columns if column not in selected_columns[split]]
    columns_to_drop_report.append({
        "Датасет": split,
        "Оставляем": selected_columns[split],
        "Удаляем": columns_to_drop,
    })

pd.DataFrame(columns_to_drop_report)


,Датасет,Оставляем,Удаляем
0,train,"[image_id_ext, image, result]","[item_id, label, ratio]"
1,val,"[image_id_ext, image, result]","[item_id, label, ratio]"
2,test,"[image_id_ext, image, item_id]","[perform_top_microcat_name, perform_top_microc..."


In [5]:
def clean_df(df: pd.DataFrame, split: str) -> pd.DataFrame:
    """Очищает train/val: оставляет нужные колонки и удаляет непригодные строки."""
    rows_before = len(df)
    columns_before = list(df.columns)

    print(f'Датасет: {split}')
    print("Размер датасета до: ", df.shape)

    cleaned_df = df[selected_columns[split]].copy()
    cleaned_df["image_id_ext"] = cleaned_df["image_id_ext"].astype(str).str.strip()
    cleaned_df["result"] = pd.to_numeric(cleaned_df["result"], errors="coerce")

    # Проверяем наличие локальной картинки
    image_exists = cleaned_df["image_id_ext"].map(lambda image_id: (image_roots[split] / f"{image_id}.jpg").exists())
    valid_rows = (
        cleaned_df["image_id_ext"].ne("")
        & cleaned_df["result"].notna()
        & (cleaned_df["result"] != 18)
        & image_exists
    )

    cleaned_df = cleaned_df.loc[valid_rows].copy()
    rows_after_filter = len(cleaned_df)

    # Удаляем полные дубликаты и повторяющиеся image_id_ext
    duplicate_rows_count = int(cleaned_df.duplicated().sum())
    cleaned_df = cleaned_df.drop_duplicates().copy()
    duplicate_image_id_count = int(cleaned_df.duplicated(subset=["image_id_ext"]).sum())
    cleaned_df = cleaned_df.drop_duplicates(subset=["image_id_ext"]).copy()

    cleaned_df["result"] = cleaned_df["result"].astype(int)


    print("Удалено полных дублей: ", duplicate_rows_count)
    print("Удалено дублей image_id_ext: ", duplicate_image_id_count)
    print("Удалено строк: ", rows_before - rows_after_filter)
    print("Удалено колонок: ", len([column for column in columns_before if column not in selected_columns[split]]))
    print("Размер датасета после: ", cleaned_df.shape, '\n')
    return cleaned_df

# Для test удаляем только лишние колонки, но сохраняем исходное количество строк.
def clean_test_df(df: pd.DataFrame) -> pd.DataFrame:
    cleaned_df = df[selected_columns["test"]].copy()
    print('Датасет: test')
    print("Размер датасета до: ", df.shape)
    print("Размер датасета после: ", cleaned_df.shape)
    return cleaned_df


train_df = clean_df(train_df, "train")
val_df = clean_df(val_df, "val")
test_df = clean_test_df(test_df)

dfs = {
    "train": train_df,
    "val": val_df,
    "test": test_df,
}


Датасет: train
Размер датасета до:  (4562, 6)
Удалено полных дублей:  0
Удалено дублей image_id_ext:  0
Удалено строк:  251
Удалено колонок:  3
Размер датасета после:  (4311, 3) 

Датасет: val
Размер датасета до:  (500, 6)
Удалено полных дублей:  0
Удалено дублей image_id_ext:  0
Удалено строк:  23
Удалено колонок:  3
Размер датасета после:  (477, 3) 

Датасет: test
Размер датасета до:  (48003, 7)
Размер датасета после:  (48003, 3)


## 4. Распределение классов после очистки

После удаления лишних колонок, непригодных строк, класса `18` и дубликатов проверяем итоговое распределение классов в `train` и `val`. Именно это распределение будет использоваться при обучении и оценке модели.


In [6]:
class_distribution_after_cleaning = []

for split in ["train", "val"]:
    counts = dfs[split]["result"].value_counts().sort_index()
    total = len(dfs[split])

    for class_id, count in counts.items():
        class_distribution_after_cleaning.append({
            "Класс": class_id,
            "Датасет": split,
            "Количество": int(count),
            "Доля": round(count / total, 4),
        })

class_distribution_after_cleaning_df = pd.DataFrame(class_distribution_after_cleaning)
class_distribution_after_cleaning_df.pivot(
    index="Класс",
    columns="Датасет",
    values="Количество",
).fillna(0).astype(int)


Датасет,train,val
Класс,,
0,248,34
1,199,15
2,247,23
3,249,30
4,251,24
5,74,21
6,215,31
7,255,23
8,255,28


Классы распределены неравномерно: часть классов встречается заметно реже остальных. Нужна балансировка. Сделаем ее опциональной

Базовый вариант - использовать веса классов в `CrossEntropyLoss`: редкие классы получают больший вес в функции потерь. Альтернативный вариант — `WeightedRandomSampler`, который чаще подаёт редкие классы в батчи. Оба варианта лучше держать как параметр обучения, чтобы можно было сравнить качество с балансировкой и без неё
